# 02 - Feature Exploration
Explore content and collaborative features.

In [1]:
import numpy as np
import scipy.sparse as sp
import os
from typing import Iterator, Tuple

class ChunkGenerator:
    def __init__(self, matrix_path: str, chunk_size: int = 1000):
        """
        Khởi tạo Iterator cắt lát ma trận thưa.
        
        Args:
            matrix_path (str): Đường dẫn đến file ma trận tfidf_matrix.npz.
            chunk_size (int): Số lượng hàng (video) tối đa được xử lý trong một lô.
        """
        self.matrix_path = matrix_path
        self.chunk_size = chunk_size
        
        # Load ma trận thưa dạng CSR (Compressed Sparse Row) để tối ưu việc cắt lát theo hàng.
        print(f"[ChunkGenerator] Đang tải ma trận thưa từ '{self.matrix_path}'...")
        if not os.path.exists(self.matrix_path):
            raise FileNotFoundError(f"Không tìm thấy file {self.matrix_path}")
            
        self.matrix = sp.load_npz(self.matrix_path)
        
        # Ép kiểu về dạng CSR nếu file lưu ở định dạng khác (như COO, CSC)
        if not isinstance(self.matrix, sp.csr_matrix):
            self.matrix = self.matrix.tocsr()
            
        self.num_rows = self.matrix.shape[0]
        self.num_features = self.matrix.shape[1]
        print(f"[ChunkGenerator] Tải thành công! Kích thước (Videos x Features): {self.matrix.shape}")

    def __iter__(self) -> Iterator[Tuple[int, int, sp.csr_matrix]]:
        """
        Generator yield từng chunk.
        
        Yields:
            start_idx (int): Chỉ số bắt đầu của lô.
            end_idx (int): Chỉ số kết thúc của lô.
            chunk (sp.csr_matrix): Ma trận con chứa 'chunk_size' hàng.
        """
        for start_idx in range(0, self.num_rows, self.chunk_size):
            end_idx = min(start_idx + self.chunk_size, self.num_rows)
            # Thao tác slicing trên hàng của CSR matrix là O(1) hoặc cực kỳ nhanh
            chunk = self.matrix[start_idx:end_idx, :]
            yield start_idx, end_idx, chunk

    def get_full_matrix(self) -> sp.csr_matrix:
        """Hàm hỗ trợ trả về toàn bộ ma trận (dùng cho Similarity Engine)."""
        return self.matrix


# ==========================================
# PHẦN MÃ CHẠY THỬ (UNIT TEST)
# ==========================================
def test_chunk_generator():
    print("--- BẮT ĐẦU TEST CHUNK GENERATOR ---")
    test_file = "test_tfidf.npz"
    
    try:
        # 1. Tạo ma trận thưa giả lập (Mock Data)
        # Giả sử ta có 5250 videos, 1000 features. Độ phủ (density) là 1%
        print("[Test] Đang tạo Mock Data...")
        mock_dense = np.random.binomial(1, 0.01, size=(5250, 1000)).astype(np.float32)
        mock_sparse = sp.csr_matrix(mock_dense)
        sp.save_npz(test_file, mock_sparse)
        
        # 2. Khởi tạo hệ thống
        chunk_size = 1000
        generator = ChunkGenerator(matrix_path=test_file, chunk_size=chunk_size)
        
        # 3. Duyệt qua vòng lặp
        total_rows_processed = 0
        for i, (start, end, chunk) in enumerate(generator):
            print(f"  -> Lô {i+1:02d}: Hàng [{start:04d} : {end:04d}] | Kích thước: {chunk.shape} | Định dạng: {type(chunk).__name__}")
            total_rows_processed += chunk.shape[0]
            
        print(f"[Test] Tổng số hàng đã xử lý: {total_rows_processed}/{generator.num_rows}")
        assert total_rows_processed == generator.num_rows, "Lỗi: Số hàng xử lý không khớp với tổng số hàng!"
        print("--- PASS: TEST CHUNK GENERATOR THÀNH CÔNG ---")
        
    finally:
        # Dọn dẹp file rác sau khi test
        if os.path.exists(test_file):
            os.remove(test_file)

if __name__ == "__main__":
    test_chunk_generator()

--- BẮT ĐẦU TEST CHUNK GENERATOR ---
[Test] Đang tạo Mock Data...
[ChunkGenerator] Đang tải ma trận thưa từ 'test_tfidf.npz'...
[ChunkGenerator] Tải thành công! Kích thước (Videos x Features): (5250, 1000)
  -> Lô 01: Hàng [0000 : 1000] | Kích thước: (1000, 1000) | Định dạng: csr_matrix
  -> Lô 02: Hàng [1000 : 2000] | Kích thước: (1000, 1000) | Định dạng: csr_matrix
  -> Lô 03: Hàng [2000 : 3000] | Kích thước: (1000, 1000) | Định dạng: csr_matrix
  -> Lô 04: Hàng [3000 : 4000] | Kích thước: (1000, 1000) | Định dạng: csr_matrix
  -> Lô 05: Hàng [4000 : 5000] | Kích thước: (1000, 1000) | Định dạng: csr_matrix
  -> Lô 06: Hàng [5000 : 5250] | Kích thước: (250, 1000) | Định dạng: csr_matrix
[Test] Tổng số hàng đã xử lý: 5250/5250
--- PASS: TEST CHUNK GENERATOR THÀNH CÔNG ---


In [2]:
import numpy as np
import scipy.sparse as sp
import time
from typing import Iterator, Tuple
###
# Engine này làm gì?

# 👉 Với mỗi vector trong chunk:

# So sánh với toàn bộ dataset
# Trả về điểm giống nhau
###

class SimilarityEngine:
    def __init__(self, full_matrix: sp.csr_matrix, is_l2_normalized: bool = True):
        """
        Khởi tạo Engine tính toán độ tương đồng.
        
        Args:
            full_matrix (sp.csr_matrix): Toàn bộ ma trận TF-IDF gốc (kích thước N x Features).
            is_l2_normalized (bool): Nếu True, Cosine Similarity = Dot Product. 
                                     Mặc định các thư viện như sklearn TfidfVectorizer đều đã chuẩn hóa L2.
        """
        self.full_matrix = full_matrix
        self.is_l2_normalized = is_l2_normalized
        
        # [TỐI ƯU HÓA HIỆU NĂNG]
        # Chuyển vị ma trận gốc 1 lần duy nhất và lưu ở định dạng CSC.
        # Phép nhân: chunk(CSR) * full_matrix_T(CSC) là tối ưu nhất cho đại số tuyến tính thưa.
        print("[SimilarityEngine] Đang khởi tạo Cache: Ma trận chuyển vị CSC (N x Features) -> (Features x N)...")
        start_time = time.time()
        self.full_matrix_T = self.full_matrix.transpose().tocsc()
        print(f"[SimilarityEngine] Hoàn tất Cache trong {time.time() - start_time:.4f}s.")

    def compute(self, chunk: sp.csr_matrix) -> np.ndarray:
        """
        Thực hiện tính Cosine Similarity cho một lô (chunk).
        
        Args:
            chunk (sp.csr_matrix): Lô ma trận TF-IDF hiện tại (chunk_size x Features).
            
        Returns:
            np.ndarray: Ma trận DÀY (Dense Matrix) chứa điểm similarity. 
                        Kích thước: (chunk_size x N). 
                        Ví dụ: (1000 x 46625) ~ tốn khoảng 186 MB RAM.
        """
        # 1. Tính Tích vô hướng thưa (Sparse Dot Product)
        # Kết quả trả về sim_sparse vẫn là dạng thưa.
        sim_sparse = chunk.dot(self.full_matrix_T)
        
        # 2. Ép kiểu về ma trận dày (Dense) trên RAM
        # Bắt buộc thực hiện ở đây vì Block sau (Top-K) dùng np.argpartition trên Dense array sẽ nhanh hơn rất nhiều.
        sim_dense = sim_sparse.toarray().astype(np.float32)
        
        # Nếu chưa chuẩn hóa L2, cần chia cho norm (Thường ít dùng với TF-IDF chuẩn)
        if not self.is_l2_normalized:
            raise NotImplementedError("Pipeline hiện tại chỉ hỗ trợ ma trận đã L2-normalized để đảm bảo tốc độ O(1).")
            
        return sim_dense

    def compute_all_chunks(self, chunk_size: int = 1000) -> Iterator[Tuple[int, int, np.ndarray]]:
        """
        Tính similarity cho toàn bộ full_matrix theo từng chunk.
        
        Args:
            chunk_size (int): Số dòng xử lý mỗi lần.
            
        Yields:
            Tuple[start_idx, end_idx, similarity_chunk]
        """
        num_rows = self.full_matrix.shape[0]
        for start_idx in range(0, num_rows, chunk_size):
            end_idx = min(start_idx + chunk_size, num_rows)
            chunk = self.full_matrix[start_idx:end_idx, :]
            similarity_chunk = self.compute(chunk)
            print(f"Kích thước ma trận kết quả: {similarity_chunk.shape}")
            print(f"[Test] Dung lượng RAM của kết quả này: {similarity_chunk.nbytes / (1024**2):.2f} MB")
            yield start_idx, end_idx, similarity_chunk

# ==========================================
# PHẦN MÃ CHẠY THỬ (INTEGRATION TEST BLOCK 1 & 2)
# ==========================================
def test_similarity_engine():
    print("--- BẮT ĐẦU TEST SIMILARITY ENGINE ---")
    
    # 1. Tạo Mock Data (Giả lập tập dữ liệu nhỏ gọn: 5250 video x 1000 features)
    num_videos = 5250
    num_features = 1000
    chunk_size = 1000
    
    print("[Test] Khởi tạo ma trận thưa giả lập...")
    mock_dense = np.random.binomial(1, 0.05, size=(num_videos, num_features)).astype(np.float32)
    full_matrix_mock = sp.csr_matrix(mock_dense)
    
    # 2. Khởi tạo Engine
    engine = SimilarityEngine(full_matrix=full_matrix_mock)
    
    # 3. Chạy giả lập 1 chunk duy nhất lấy từ full_matrix
    chunk_mock = full_matrix_mock[0:chunk_size, :]
    print(f"[Test] Truyền vào Chunk có shape: {chunk_mock.shape}")
    
    start_compute = time.time()
    similarity_scores = engine.compute(chunk_mock)
    compute_time = time.time() - start_compute
    
    # 4. Kiểm tra kết quả
    print(f"[Test] Kích thước ma trận kết quả: {similarity_scores.shape}")
    print(f"[Test] Dung lượng RAM của kết quả này: {similarity_scores.nbytes / (1024**2):.2f} MB")
    print(f"[Test] Thời gian tính toán cho 1 chunk: {compute_time:.4f}s")
    
    assert similarity_scores.shape == (chunk_size, num_videos), "Lỗi: Kích thước ma trận điểm số không đúng!"
    assert similarity_scores.dtype == np.float32, "Lỗi: Kiểu dữ liệu phải là float32 để tiết kiệm RAM!"
    print("--- PASS: TEST SIMILARITY ENGINE THÀNH CÔNG ---")

if __name__ == "__main__":
    test_similarity_engine()

--- BẮT ĐẦU TEST SIMILARITY ENGINE ---
[Test] Khởi tạo ma trận thưa giả lập...
[SimilarityEngine] Đang khởi tạo Cache: Ma trận chuyển vị CSC (N x Features) -> (Features x N)...
[SimilarityEngine] Hoàn tất Cache trong 0.0000s.
[Test] Truyền vào Chunk có shape: (1000, 1000)
[Test] Kích thước ma trận kết quả: (1000, 5250)
[Test] Dung lượng RAM của kết quả này: 20.03 MB
[Test] Thời gian tính toán cho 1 chunk: 0.1283s
--- PASS: TEST SIMILARITY ENGINE THÀNH CÔNG ---


In [3]:
import numpy as np
import time
from typing import List, Dict, Tuple

class TopKExtractor:
    def __init__(self, video_ids: np.ndarray, k: int = 50):
        """
        Khởi tạo bộ trích xuất Top-K.
        
        Args:
            video_ids (np.ndarray): Mảng 1D chứa chuỗi video_id thực tế, độ dài N.
                                    Dùng để map từ index (0 -> N-1) sang ID thật.
            k (int): Số lượng video tương đồng cần giữ lại (Top-K).
        """
        self.video_ids = video_ids
        self.k = k
        self.num_videos = len(video_ids)

    def extract(self, similarity_chunk: np.ndarray, chunk_start_idx: int) -> Dict[str, List[Tuple[str, float]]]:
        """
        Trích xuất Top-K on-the-fly từ ma trận điểm số của một lô.
        
        Args:
            similarity_chunk (np.ndarray): Ma trận điểm số của lô hiện tại (chunk_size x N).
            chunk_start_idx (int): Chỉ số dòng bắt đầu của lô này trong ma trận tổng.
            
        Returns:
            Dict[str, List[Tuple[str, float]]]: Dictionary map từ 'source_video_id' 
                                                sang danh sách 50 tuple (similar_video_id, score).
        """
        batch_size = similarity_chunk.shape[0]
        
        # Đảm bảo K không lớn hơn tổng số lượng video hiện có
        actual_k = min(self.k, self.num_videos)
        
        # 1. PHÂN TÁCH MẢNG (O(N)): Lấy index của K phần tử lớn nhất (chưa sắp xếp)
        # Sử dụng -actual_k để lấy k phần tử lớn nhất đẩy về cuối mảng
        partitioned_idx = np.argpartition(similarity_chunk, -actual_k, axis=1)[:, -actual_k:]
        
        # 2. Lấy điểm số tương ứng của K phần tử này
        top_k_scores = np.take_along_axis(similarity_chunk, partitioned_idx, axis=1)
        
        # 3. Sắp xếp lại cục bộ K phần tử này theo thứ tự giảm dần (O(K log K))
        # np.argsort sắp xếp tăng dần, nên truyền dấu âm vào top_k_scores để sort giảm dần
        sorted_k_relative_idx = np.argsort(-top_k_scores, axis=1)
        
        # 4. Gióng ngược lại để lấy index và score cuối cùng (đã sắp xếp)
        final_top_k_idx = np.take_along_axis(partitioned_idx, sorted_k_relative_idx, axis=1)
        final_top_k_scores = np.take_along_axis(top_k_scores, sorted_k_relative_idx, axis=1)
        
        # 5. Xây dựng cấu trúc dữ liệu kết quả chuẩn bị cho Key-Value Store
        results = {}
        for i in range(batch_size):
            global_video_idx = chunk_start_idx + i
            source_video_id = self.video_ids[global_video_idx]
            
            # Zip id và score (Ép kiểu float gốc của Python để dễ serialize thành JSON/Pickle sau này)
            similar_items = [
                (self.video_ids[idx], float(score)) 
                for idx, score in zip(final_top_k_idx[i], final_top_k_scores[i])
            ]
            results[source_video_id] = similar_items
            
        return results

# ==========================================
# PHẦN MÃ CHẠY THỬ (INTEGRATION TEST BLOCK 2 & 3)
# ==========================================
def test_top_k_extractor():
    print("--- BẮT ĐẦU TEST TOP-K EXTRACTOR ---")
    
    # 1. Giả lập dữ liệu đầu vào (từ Similarity Engine)
    chunk_size = 1000
    num_videos = 46625
    k = 50
    
    # Tạo danh sách video_id giả lập: "video_00000", "video_00001", ...
    print("[Test] Khởi tạo mảng Video IDs...")
    mock_video_ids = np.array([f"video_{str(i).zfill(5)}" for i in range(num_videos)])
    
    # Giả lập ma trận điểm số sau khi nhân Cosine Similarity (DENSE MATRIX)
    print(f"[Test] Giả lập similarity_chunk kích thước ({chunk_size} x {num_videos})...")
    np.random.seed(42) # Set seed để kết quả cố định
    mock_similarity_chunk = np.random.rand(chunk_size, num_videos).astype(np.float32)
    
    # 2. Khởi tạo Extractor
    extractor = TopKExtractor(video_ids=mock_video_ids, k=k)
    
    # 3. Đo lường hiệu năng xử lý (Performance Benchmark)
    print("[Test] Bắt đầu trích xuất Top-K...")
    start_extract = time.time()
    
    # Giả định chunk này nằm từ dòng 0 đến 1000 trong ma trận tổng
    results = extractor.extract(similarity_chunk=mock_similarity_chunk, chunk_start_idx=0)
    
    extract_time = time.time() - start_extract
    
    # 4. Kiểm tra kết quả
    sample_key = "video_00000"
    print(f"[Test] Thời gian trích xuất Top-{k} cho {chunk_size} dòng x {num_videos} cột: {extract_time:.4f}s")
    print(f"[Test] Số lượng keys trong kết quả: {len(results)}")
    print(f"[Test] Sample - 3 video tương đồng nhất cho '{sample_key}':")
    for vid, score in results[sample_key][:3]:
        print(f"       + ID: {vid} | Score: {score:.4f}")
        
    assert len(results) == chunk_size, "Lỗi: Số lượng keys đầu ra không khớp với chunk size!"
    assert len(results[sample_key]) == k, f"Lỗi: Số lượng video trích xuất không đúng {k}!"
    print("--- PASS: TEST TOP-K EXTRACTOR THÀNH CÔNG ---")

if __name__ == "__main__":
    test_top_k_extractor()

--- BẮT ĐẦU TEST TOP-K EXTRACTOR ---
[Test] Khởi tạo mảng Video IDs...
[Test] Giả lập similarity_chunk kích thước (1000 x 46625)...
[Test] Bắt đầu trích xuất Top-K...
[Test] Thời gian trích xuất Top-50 cho 1000 dòng x 46625 cột: 0.2269s
[Test] Số lượng keys trong kết quả: 1000
[Test] Sample - 3 video tương đồng nhất cho 'video_00000':
       + ID: video_43676 | Score: 1.0000
       + ID: video_15618 | Score: 0.9999
       + ID: video_21043 | Score: 0.9999
--- PASS: TEST TOP-K EXTRACTOR THÀNH CÔNG ---


In [4]:
import os
import pickle
import json
import time
from typing import List, Dict, Tuple

class HybridRecommendationHook:
    def __init__(self, k_final: int = 50, rrf_k: int = 60):
        """
        Khởi tạo cổng nối Hybrid sử dụng Reciprocal Rank Fusion (RRF).
        
        Args:
            k_final (int): Số lượng video cuối cùng cần giữ lại sau khi trộn.
            rrf_k (int): Hằng số điều chuẩn cho RRF (thường set là 60 theo chuẩn nghiên cứu).
        """
        self.k_final = k_final
        self.rrf_k = rrf_k

    def blend(self, cb_results: Dict[str, List[Tuple[str, float]]], 
                    cf_results: Dict[str, List[Tuple[str, float]]] = None) -> Dict[str, List[Tuple[str, float]]]:
        """
        Hợp nhất kết quả từ Content-based và Collaborative Filtering (tương lai).
        """
        # [NGÀY 4 - PASS-THROUGH]: Hiện tại chưa có CF, chỉ cho CB đi qua
        if cf_results is None:
            return cb_results
            
        # [TƯƠNG LAI - CHUẨN BỊ SẴN LOGIC HYBRID]
        hybrid_results = {}
        for source_video in cb_results.keys():
            rrf_scores = {}
            
            # 1. Tính điểm RRF cho Content-based (Dựa trên thứ hạng/index)
            for rank, (vid, _) in enumerate(cb_results.get(source_video, [])):
                rrf_scores[vid] = rrf_scores.get(vid, 0.0) + 1.0 / (self.rrf_k + rank + 1)
                
            # 2. Tính điểm RRF cho Collaborative Filtering
            for rank, (vid, _) in enumerate(cf_results.get(source_video, [])):
                rrf_scores[vid] = rrf_scores.get(vid, 0.0) + 1.0 / (self.rrf_k + rank + 1)
                
            # 3. Sắp xếp lại theo điểm RRF giảm dần và cắt Top-K
            sorted_hybrid = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)[:self.k_final]
            hybrid_results[source_video] = sorted_hybrid
            
        return hybrid_results


class KeyValueStore:
    def __init__(self, output_path: str, format: str = 'pkl'):
        """
        Khởi tạo hệ thống ghi Key-Value.
        
        Args:
            output_path (str): Nơi lưu file kết quả.
            format (str): Định dạng lưu trữ ('pkl' hoặc 'json'). Khuyến nghị 'pkl' cho tốc độ load O(1).
        """
        self.output_path = output_path
        self.format = format.lower()
        self.global_store: Dict[str, List[Tuple[str, float]]] = {}
        
        if self.format not in ['pkl', 'json']:
            raise ValueError("Chỉ hỗ trợ định dạng 'pkl' hoặc 'json'.")

    def update_batch(self, batch_data: Dict[str, List[Tuple[str, float]]]):
        """Cập nhật dữ liệu của một lô vào bộ nhớ tổng."""
        self.global_store.update(batch_data)

    def flush_to_disk(self):
        """Ghi toàn bộ dữ liệu từ RAM xuống đĩa cứng."""
        print(f"[KeyValueStore] Bắt đầu ghi {len(self.global_store)} bản ghi xuống file '{self.output_path}'...")
        start_time = time.time()
        
        if self.format == 'pkl':
            with open(self.output_path, 'wb') as f:
                pickle.dump(self.global_store, f, protocol=pickle.HIGHEST_PROTOCOL)
        elif self.format == 'json':
            with open(self.output_path, 'w', encoding='utf-8') as f:
                json.dump(self.global_store, f, ensure_ascii=False)
                
        print(f"[KeyValueStore] Ghi file thành công trong {time.time() - start_time:.4f}s.")


# ==========================================
# PHẦN MÃ CHẠY THỬ (INTEGRATION TEST BLOCK 4 & 5)
# ==========================================
def test_output_pipeline():
    print("--- BẮT ĐẦU TEST HYBRID HOOK & KV STORE ---")
    
    # 1. Giả lập kết quả đầu ra từ Block 3 (Top-K Extractor)
    print("[Test] Giả lập dữ liệu từ Content-based Pipeline...")
    mock_cb_results = {
        "video_001": [("video_100", 0.95), ("video_101", 0.88), ("video_102", 0.76)],
        "video_002": [("video_200", 0.92), ("video_201", 0.81), ("video_202", 0.70)]
    }
    
    # [Kiểm thử mở rộng]: Giả lập dữ liệu CF trong tương lai để test logic RRF
    mock_cf_results = {
        "video_001": [("video_105", 0.99), ("video_100", 0.85)], # video_100 xuất hiện ở cả 2 nguồn
        "video_002": [("video_205", 0.90)]
    }
    
    # 2. Khởi tạo Hook và thử nghiệm Blend
    hook = HybridRecommendationHook(k_final=3)
    
    # Cửa ải 1: Chỉ có CB (Ngày 4)
    day_4_results = hook.blend(cb_results=mock_cb_results, cf_results=None)
    assert day_4_results == mock_cb_results, "Lỗi: Pass-through layer hoạt động không đúng!"
    print("[Test] Pass-through Layer (Ngày 4) hoạt động hoàn hảo.")
    
    # Cửa ải 2: Có cả CB và CF (Tương lai)
    future_results = hook.blend(cb_results=mock_cb_results, cf_results=mock_cf_results)
    print("[Test] Demo Hybrid Blending (RRF) cho 'video_001':")
    for vid, rrf_score in future_results["video_001"]:
        print(f"       + ID: {vid} | RRF Score: {rrf_score:.6f}")
    
    # 3. Khởi tạo KV Store và ghi đĩa
    output_file = "test_recommendations.pkl"
    store = KeyValueStore(output_path=output_file, format='pkl')
    
    # Cập nhật lô dữ liệu và flush
    store.update_batch(day_4_results)
    store.flush_to_disk()
    
    # 4. Kiểm tra file và Load thử O(1)
    file_size_kb = os.path.getsize(output_file) / 1024
    print(f"[Test] Dung lượng file lưu trữ: {file_size_kb:.2f} KB")
    
    print("[Test] Giả lập API Backend nạp file vào RAM...")
    with open(output_file, 'rb') as f:
        loaded_lookup_table = pickle.load(f)
        
    assert "video_001" in loaded_lookup_table, "Lỗi: Key không tồn tại trong file đã lưu!"
    print(f"[Test] API Query O(1) cho 'video_001': {loaded_lookup_table['video_001']}")
    
    print("--- PASS: TEST OUTPUT PIPELINE THÀNH CÔNG ---")
    
    # Dọn dẹp
    if os.path.exists(output_file):
        os.remove(output_file)

if __name__ == "__main__":
    test_output_pipeline()

--- BẮT ĐẦU TEST HYBRID HOOK & KV STORE ---
[Test] Giả lập dữ liệu từ Content-based Pipeline...
[Test] Pass-through Layer (Ngày 4) hoạt động hoàn hảo.
[Test] Demo Hybrid Blending (RRF) cho 'video_001':
       + ID: video_100 | RRF Score: 0.032522
       + ID: video_105 | RRF Score: 0.016393
       + ID: video_101 | RRF Score: 0.016129
[KeyValueStore] Bắt đầu ghi 2 bản ghi xuống file 'test_recommendations.pkl'...
[KeyValueStore] Ghi file thành công trong 0.0010s.
[Test] Dung lượng file lưu trữ: 0.18 KB
[Test] Giả lập API Backend nạp file vào RAM...
[Test] API Query O(1) cho 'video_001': [('video_100', 0.95), ('video_101', 0.88), ('video_102', 0.76)]
--- PASS: TEST OUTPUT PIPELINE THÀNH CÔNG ---


# Main

In [5]:
from pathlib import Path
if __name__ == "__main__":
    ##########Chunking###########
    # Data: thử nhiều relative path vì cwd của notebook có thể là project root hoặc thư mục notebooks
    candidates = [
        Path("data/processed/tfidf_matrix.npz"),
        Path("../data/processed/tfidf_matrix.npz"),
    ]
    
    file_path = None
    for p in candidates:
        if p.exists():
            file_path = str(p)
            break
    
    if file_path is None:
        cwd = Path.cwd()
        raise FileNotFoundError(
            f"Không tìm thấy file tfidf_matrix.npz. CWD hiện tại: {cwd}. "
            "Đã thử: data/processed/tfidf_matrix.npz và ../data/processed/tfidf_matrix.npz"
        )
    
    print(f"[Main] Dùng file: {file_path}")
    
    # 2. Khởi tạo hệ thống
    chunk_size = 1000
    generator = ChunkGenerator(matrix_path=file_path, chunk_size=chunk_size)
    
    # 3. Duyệt qua vòng lặp
    total_rows_processed = 0
    for i, (start, end, chunk) in enumerate(generator):
        print(f"  -> Lô {i+1:02d}: Hàng [{start:04d} : {end:04d}] | Kích thước: {chunk.shape} | Định dạng: {type(chunk).__name__}")
        total_rows_processed += chunk.shape[0]
        
    print(f"[Test] Tổng số hàng đã xử lý: {total_rows_processed}/{generator.num_rows}")
    assert total_rows_processed == generator.num_rows, "Lỗi: Số hàng xử lý không khớp với tổng số hàng!"
    print("--- PASS: TEST CHUNK GENERATOR THÀNH CÔNG ---")
    #####SimilarityEngine########
    # 2. Khởi tạo Engine
    full_matrix_real = generator.get_full_matrix()
    engine = SimilarityEngine(full_matrix=full_matrix_real)
    
    # 3. Chạy giả lập 1 chunk duy nhất lấy từ full_matrix
    chunk_mock = full_matrix_real[0:chunk_size, :]
    print(f"[Test] Truyền vào Chunk có shape: {chunk_mock.shape}")
    
    similarity_chunks = engine.compute_all_chunks(chunk_size=chunk_size)
    print(similarity_chunks)

[Main] Dùng file: ..\data\processed\tfidf_matrix.npz
[ChunkGenerator] Đang tải ma trận thưa từ '..\data\processed\tfidf_matrix.npz'...
[ChunkGenerator] Tải thành công! Kích thước (Videos x Features): (46625, 10000)
  -> Lô 01: Hàng [0000 : 1000] | Kích thước: (1000, 10000) | Định dạng: csr_matrix
  -> Lô 02: Hàng [1000 : 2000] | Kích thước: (1000, 10000) | Định dạng: csr_matrix
  -> Lô 03: Hàng [2000 : 3000] | Kích thước: (1000, 10000) | Định dạng: csr_matrix
  -> Lô 04: Hàng [3000 : 4000] | Kích thước: (1000, 10000) | Định dạng: csr_matrix
  -> Lô 05: Hàng [4000 : 5000] | Kích thước: (1000, 10000) | Định dạng: csr_matrix
  -> Lô 06: Hàng [5000 : 6000] | Kích thước: (1000, 10000) | Định dạng: csr_matrix
  -> Lô 07: Hàng [6000 : 7000] | Kích thước: (1000, 10000) | Định dạng: csr_matrix
  -> Lô 08: Hàng [7000 : 8000] | Kích thước: (1000, 10000) | Định dạng: csr_matrix
  -> Lô 09: Hàng [8000 : 9000] | Kích thước: (1000, 10000) | Định dạng: csr_matrix
  -> Lô 10: Hàng [9000 : 10000] | Kích

In [8]:
num_videos = full_matrix_real.shape[0]
num_features = 10000
chunk_size = 1000
# 3. Chạy giả lập 1 chunk duy nhất lấy từ full_matrix
chunk_mock = full_matrix_real[0:chunk_size, :]
print(f"[Test] Truyền vào Chunk có shape: {chunk_mock.shape}")

start_compute = time.time()
similarity_scores = engine.compute(chunk_mock)
compute_time = time.time() - start_compute

# 4. Kiểm tra kết quả
print(f"[Test] Kích thước ma trận kết quả: {similarity_scores.shape}")
print(f"[Test] Dung lượng RAM của kết quả này: {similarity_scores.nbytes / (1024**2):.2f} MB")
print(f"[Test] Thời gian tính toán cho 1 chunk: {compute_time:.4f}s")

assert similarity_scores.shape == (chunk_size, num_videos), "Lỗi: Kích thước ma trận điểm số không đúng!"
assert similarity_scores.dtype == np.float32, "Lỗi: Kiểu dữ liệu phải là float32 để tiết kiệm RAM!"
print("--- PASS: TEST SIMILARITY ENGINE THÀNH CÔNG ---")


engine.


[Test] Truyền vào Chunk có shape: (1000, 10000)
[Test] Kích thước ma trận kết quả: (1000, 46625)
[Test] Dung lượng RAM của kết quả này: 177.86 MB
[Test] Thời gian tính toán cho 1 chunk: 0.4300s
--- PASS: TEST SIMILARITY ENGINE THÀNH CÔNG ---


In [ ]:
from pathlib import Path
import pickle

video_ids = np.array([f"video_{str(i).zfill(5)}" for i in range(num_videos)])
extractor = TopKExtractor(video_ids=video_ids, k=50)
all_top_k_results = {}

for start_idx, end_idx, chunk in generator:
    print(f"Đang xử lý từ {start_idx} đến {end_idx}")
    similarity_scores = engine.compute(chunk)
    print(f"Kích thước ma trận kết quả: {similarity_scores.shape}")
    # Trích xuất Top-K video tương tự
    top_k_results = extractor.extract(similarity_scores, chunk_start_idx=start_idx)
    all_top_k_results.update(top_k_results)
    print(f"Top-50 video tương tự:")
    for source_video, similars in list(top_k_results.items())[:3]:  # In ra 3 video đầu tiên để demo
        print(f"  - {source_video}:")
        for similar_video, score in similars[:5]:  # In ra 5 video tương tự đầu tiên
            print(f"      + {similar_video} (Score: {score:.4f})")

project_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "artifacts").exists()),
    Path.cwd(),
)
output_dir = project_root / "artifacts"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "top_k_results.pkl"

with open(output_path, "wb") as f:
    pickle.dump(all_top_k_results, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Đã lưu toàn bộ kết quả Top-K vào: {output_path}")
print(f"Tổng số video đã lưu: {len(all_top_k_results)}")

Đang xử lý từ 0 đến 1000
Kích thước ma trận kết quả: (1000, 46625)


IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

46625
